In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random
import pickle
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size = 64
block_size = 128
max_iters = 3000
learning_rate = 3e-4
eval_iters = 500
n_embd = 384
n_layer = 8
n_head = 8
dropout = 0.2

cuda


In [3]:
chars = ""
with open('vocab.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)

In [4]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda x: [string_to_int[c] for c in x]
decode = lambda y: ''.join(int_to_string[i] for i in y)

data =  torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 0,  0,  1,  0,  2,  0,  3,  0,  4,  0,  5,  0,  6,  0,  7,  0,  8,  0,
         9,  0, 10,  0, 11,  0, 12,  0, 13,  0, 14,  0, 15,  0, 16,  0, 17,  0,
        18,  0, 19,  0, 20,  0, 21,  0, 22,  0, 23,  0, 24,  0, 25,  0, 26,  0,
        27,  0, 28,  0, 29,  0, 30,  0, 31,  0, 32,  0, 33,  0, 34,  0, 35,  0,
        36,  0, 37,  0, 38,  0, 39,  0, 40,  0, 41,  0, 42,  0, 43,  0, 44,  0,
        45,  0, 46,  0, 47,  0, 48,  0, 49,  0])


In [5]:
length = int(0.8*len(data))
training_portion = data[:length]
val_data = data[length:]

# memory map for using small snippets of text from a single file of any size
def get_rchunk(split):
    filename = 'output_train.txt' if split == 'train' else 'output_val.txt'
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            # determine the file size and find a random place to start reading
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size*batch_size)

            # seek to the start position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size*batch_size - 1)

            # decode the block to a string, ignoring and invalid byte sequences
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', '')

            # train and test splits
            data = torch.tensor(encode(decoded_block), dtype=torch.long)
            
    return data

def get_batch(split):
    data = get_rchunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # we comment out the print so in the training loop, it doesn't print out every tensor
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# x, y = get_batch('train')
# print('inputs:')
# # print(x.shape)
# print(x)
# print('targets:')
# print(y)

In [6]:
x = training_portion[:block_size]
y = training_portion[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([0]) target is tensor(0)
when input is tensor([0, 0]) target is tensor(1)
when input is tensor([0, 0, 1]) target is tensor(0)
when input is tensor([0, 0, 1, 0]) target is tensor(2)
when input is tensor([0, 0, 1, 0, 2]) target is tensor(0)
when input is tensor([0, 0, 1, 0, 2, 0]) target is tensor(3)
when input is tensor([0, 0, 1, 0, 2, 0, 3]) target is tensor(0)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0]) target is tensor(4)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4]) target is tensor(0)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0]) target is tensor(5)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5]) target is tensor(0)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0]) target is tensor(6)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0, 6]) target is tensor(0)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0, 6, 0]) target is tensor(7)
when input is tensor([0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0, 6, 0, 7]) target is tenso

In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
class Head(nn.Module):
    """ One head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x) # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B,T,hs) @ (B,hs,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B,T,T) @ (B,T,hs) -> (B,T,hs)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, C) -> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3]) for 3 heads
        out = self.dropout(self.proj(out))
        return out
    
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # how many decoder blocks we have running sequentially
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # logits are effectively a probability distribution of what we want to predict
        B, T = index.shape

        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # view() expects inputs of (C), (N, C), or (N, C, d_1, d_2, d_3,...,d_K) with K >= 1
            # where N is the batch size, and C is the number of classes/channels
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indeices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get the probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
if os.path.exists('model-01.pkl'):
    print('loading model parameters...')
    with open('model-01.pkl', 'rb') as f:
        model = pickle.load(f)
    print('loaded successfully')
m = model.to(device)

# context = torch.zeros((1,1), dtype=torch.long, device=device)
# generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
# print(generated_chars)

loading model parameters...
loaded successfully


In [ ]:
# now to create a pytorch optimizer (This is the training loop)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses["train"]:.4f}, validation loss: {losses["val"]:.4f}')
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print("model saved")

step: 0, train loss: 1.6135, validation loss: 1.5835


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)